# 02 — OGG Phase 1 Exploratory Data Analysis

This notebook characterizes the downloaded OGG flight, NOAA airport-weather, and Hawaii Storm Events datasets before we construct the time-aligned modeling table.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot locate project root from {cwd}')

BTS_FILE = ROOT / 'data/raw/bts/ogg_flights_2020_2026.csv.gz'
WEATHER_FILE = ROOT / 'data/raw/weather/ogg_lcd_2020_2026.csv.gz'
STORM_FILE = ROOT / 'data/raw/incidents/hawaii_storm_events_2020_2026.csv.gz'

print('ROOT:', ROOT)
print('Flights exists:', BTS_FILE.exists())
print('Weather exists:', WEATHER_FILE.exists())
print('Storms exists:', STORM_FILE.exists())


## 1. Flights: load, label, and inspect


In [ ]:
flights = pd.read_csv(BTS_FILE, low_memory=False)
flights['FlightDate'] = pd.to_datetime(flights['FlightDate'], errors='coerce')
flights['year'] = flights['FlightDate'].dt.year
flights['month'] = flights['FlightDate'].dt.month
flights['year_month'] = flights['FlightDate'].dt.to_period('M').astype(str)
flights['direction'] = np.where(flights['Origin'].eq('OGG'), 'departure', 'arrival')
flights['covid_era'] = flights['FlightDate'].between('2020-03-01', '2021-05-31')

def disruption_class(r):
    if r.get('Cancelled', 0) == 1:
        return 'cancelled'
    d = r.get('ArrDelay', np.nan)
    if pd.isna(d):
        return 'unknown'
    if d < 15:
        return 'normal'
    if d < 180:
        return 'delay'
    return 'severe_delay'

flights['disruption_class'] = flights.apply(disruption_class, axis=1)
print('Flights:', flights.shape)
display(flights.head())
display(flights['disruption_class'].value_counts(dropna=False).to_frame('count'))
display((flights['disruption_class'].value_counts(normalize=True)*100).round(2).to_frame('percent'))


In [ ]:
yearly = flights.groupby('year').agg(
    flights=('FlightDate','size'),
    cancelled=('Cancelled','sum'),
    mean_arr_delay=('ArrDelay','mean'),
    median_arr_delay=('ArrDelay','median')
)
yearly['cancel_rate_pct'] = 100 * yearly['cancelled'] / yearly['flights']
display(yearly.round(2))

airlines = flights.groupby('Reporting_Airline').agg(
    flights=('FlightDate','size'),
    cancelled=('Cancelled','sum'),
    mean_arr_delay=('ArrDelay','mean'),
    severe_delays=('disruption_class', lambda s: (s=='severe_delay').sum())
)
airlines['cancel_rate_pct'] = 100 * airlines['cancelled'] / airlines['flights']
airlines['severe_delay_rate_pct'] = 100 * airlines['severe_delays'] / airlines['flights']
display(airlines.sort_values('flights', ascending=False).round(2))

display(pd.crosstab(flights['direction'], flights['disruption_class'], normalize='index').mul(100).round(2))
display(flights.groupby('covid_era').agg(flights=('FlightDate','size'), cancelled=('Cancelled','sum'), mean_arr_delay=('ArrDelay','mean')).round(2))


In [ ]:
monthly = flights.groupby('year_month').agg(flights=('FlightDate','size'), cancelled=('Cancelled','sum'), mean_arr_delay=('ArrDelay','mean'))
monthly['cancel_rate_pct'] = 100 * monthly['cancelled'] / monthly['flights']
display(monthly.tail(30).round(2))

flights['disruption_class'].value_counts().plot(kind='bar', figsize=(8,4), title='OGG disruption classes')
plt.ylabel('Flights'); plt.tight_layout(); plt.show()
yearly['cancel_rate_pct'].plot(marker='o', figsize=(8,4), title='OGG cancellation rate by year')
plt.ylabel('Cancellation rate (%)'); plt.tight_layout(); plt.show()


## 2. NOAA airport weather: schema and missingness


In [ ]:
weather = pd.read_csv(WEATHER_FILE, low_memory=False)
print('Weather:', weather.shape)
display(weather.head())
display(pd.DataFrame({'column': weather.columns}))
missing = weather.isna().mean().mul(100).sort_values(ascending=False).rename('missing_pct').to_frame()
display(missing.head(30).round(2))
keywords = ['date','hourly','wind','gust','visibility','precip','pressure','temperature','dew','sky','weather']
candidate_cols = [c for c in weather.columns if any(k in c.lower() for k in keywords)]
display(pd.DataFrame({'candidate_weather_feature': candidate_cols}))


## 3. NOAA Hawaii Storm Events: event types and timing


In [ ]:
storms = pd.read_csv(STORM_FILE, low_memory=False)
print('Storm events:', storms.shape)
display(storms.head())
if 'EVENT_TYPE' in storms.columns:
    display(storms['EVENT_TYPE'].value_counts().head(30).to_frame('count'))
if 'YEAR' in storms.columns and 'EVENT_TYPE' in storms.columns:
    display(pd.crosstab(storms['YEAR'], storms['EVENT_TYPE']))
useful = [c for c in ['EVENT_ID','EVENT_TYPE','BEGIN_DATE_TIME','END_DATE_TIME','CZ_NAME','SOURCE','MAGNITUDE','DEATHS_DIRECT','INJURIES_DIRECT','DAMAGE_PROPERTY','DAMAGE_CROPS','BEGIN_LAT','BEGIN_LON','END_LAT','END_LON','EVENT_NARRATIVE'] if c in storms.columns]
display(storms[useful].head(20))


## 4. Broad storm-day overlap check
This first check uses any Hawaii event on the same date. Notebook 03 will refine this to Maui/OGG-specific spatial and hourly matching.


In [ ]:
if {'BEGIN_DATE_TIME','END_DATE_TIME'}.issubset(storms.columns):
    storms['begin_dt'] = pd.to_datetime(storms['BEGIN_DATE_TIME'], errors='coerce')
    storms['end_dt'] = pd.to_datetime(storms['END_DATE_TIME'], errors='coerce')
    ranges = []
    for r in storms[['begin_dt','end_dt']].dropna().itertuples(index=False):
        if r.end_dt >= r.begin_dt:
            ranges.extend(pd.date_range(r.begin_dt.normalize(), r.end_dt.normalize(), freq='D'))
    storm_dates = pd.Index(pd.Series(ranges).drop_duplicates())
    flights['storm_day_any_hawaii'] = flights['FlightDate'].dt.normalize().isin(storm_dates)
    overlap = flights.groupby('storm_day_any_hawaii').agg(flights=('FlightDate','size'), cancelled=('Cancelled','sum'), mean_arr_delay=('ArrDelay','mean'))
    overlap['cancel_rate_pct'] = 100 * overlap['cancelled'] / overlap['flights']
    display(overlap.round(2))
else:
    print('Storm datetime fields are missing; overlap deferred.')


## What to record after running

Please keep the outputs for: class balance, yearly cancellation rates, airline summary, weather shape/missingness/candidate columns, storm-event type counts, and broad storm-day overlap. These results determine the exact merge strategy and baseline model design.
